### 06 - Modele lineaire d'attrition
#### HumanForYou - Attrition ML

Objectif: entrainer un modele lineaire (Logistic Regression) sur les donnees preparees en 05, elles-memes derivees de 04.

- **Entrees**: `data/processed/attrition_train_prepared.csv`, `data/processed/attrition_test_prepared.csv`
- **Sorties**: `data/processed/attrition_linear_metrics.csv`, `data/processed/attrition_linear_test_predictions.csv`, `data/processed/attrition_linear_coefficients.csv`

#### 1. Imports

Cette section importe:
- les outils de modelisation (`LogisticRegression`),
- les metriques de classification,
- les librairies d'export/visualisation (`pandas`, `matplotlib`).

Le choix d'un modele lineaire sert de baseline interpretable pour l'attrition.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

#### 2. Chargement

On charge les jeux train/test prepares en 05, puis on separe:
- les variables explicatives (`X_train`, `X_test`),
- la cible (`y_train`, `y_test`).

Aucun nouveau nettoyage n'est applique ici pour conserver une evaluation comparable et tracable.


In [2]:
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
train_path = os.path.join(PROCESSED_DIR, 'attrition_train_prepared.csv')
test_path = os.path.join(PROCESSED_DIR, 'attrition_test_prepared.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=['Attrition']).copy()
y_train = train_df['Attrition'].astype(int).copy()
X_test = test_df.drop(columns=['Attrition']).copy()
y_test = test_df['Attrition'].astype(int).copy()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

X_train: (3528, 46) | X_test: (882, 46)


#### 3. Entrainement et evaluation

Le modele de regression logistique est entraine avec `class_weight='balanced'` pour tenir compte du desequilibre de classes.

Ensuite, on calcule:
- les probabilites d'attrition,
- les predictions binaires (seuil 0.5),
- les metriques principales (`accuracy`, `precision`, `recall`, `f1`, `roc_auc`).

On extrait aussi les coefficients pour interpreter les variables les plus influentes.


In [3]:
model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metrics_df = pd.DataFrame([
    {
        'model': 'LogisticRegression_balanced',
        'accuracy_test': float(accuracy_score(y_test, y_pred)),
        'precision_test': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall_test': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_test': float(f1_score(y_test, y_pred, zero_division=0)),
        'roc_auc_test': float(roc_auc_score(y_test, y_proba)),
    }
])

predictions_df = pd.DataFrame({
    'y_true': y_test,
    'y_proba_logistic': y_proba,
    'y_pred_logistic': y_pred,
})

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_.ravel(),
}).sort_values('coefficient', key=np.abs, ascending=False)

metrics_df

C:\git\humanforyou-attrition-ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,accuracy_test,precision_test,recall_test,f1_test,roc_auc_test
0,LogisticRegression_balanced,0.764172,0.376866,0.711268,0.492683,0.791292


#### 4. Export + figures

Les resultats sont exportes sous forme de fichiers CSV (metriques, predictions, coefficients) et de figures:
- barplot des coefficients les plus forts,
- courbe ROC sur le jeu test.

Ces sorties permettent d'evaluer a la fois la performance globale et l'interpretabilite du modele lineaire.


In [4]:
FIG_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

metrics_path = os.path.join(PROCESSED_DIR, 'attrition_linear_metrics.csv')
preds_path = os.path.join(PROCESSED_DIR, 'attrition_linear_test_predictions.csv')
coef_path = os.path.join(PROCESSED_DIR, 'attrition_linear_coefficients.csv')

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(preds_path, index=False)
coef_df.to_csv(coef_path, index=False)

top_coef = coef_df.head(15).sort_values('coefficient')
plt.figure(figsize=(9, 6))
plt.barh(top_coef['feature'], top_coef['coefficient'])
plt.title('Top 15 coefficients (Logistic Regression)')
plt.tight_layout()
coef_fig_path = os.path.join(FIG_DIR, 'attrition_linear_top_coefficients.png')
plt.savefig(coef_fig_path, dpi=300)
plt.close()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label='Logistic Regression')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Attrition')
plt.legend()
plt.tight_layout()
roc_fig_path = os.path.join(FIG_DIR, 'attrition_linear_roc_curve.png')
plt.savefig(roc_fig_path, dpi=300)
plt.close()

print(f'Saved: {metrics_path}')
print(f'Saved: {preds_path}')
print(f'Saved: {coef_path}')
print(f'Saved: {coef_fig_path}')
print(f'Saved: {roc_fig_path}')

Saved: ..\data\processed\attrition_linear_metrics.csv
Saved: ..\data\processed\attrition_linear_test_predictions.csv
Saved: ..\data\processed\attrition_linear_coefficients.csv
Saved: ..\reports\figures\attrition_linear_top_coefficients.png
Saved: ..\reports\figures\attrition_linear_roc_curve.png
